# TraceGenAI - Log Monitoring and Resolution AI Agent

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class LogState(AgentState):
    formatted_log: str
    websearch_results : str
    db_results : str
    past_incidents : str
    root_cause_analysis : str



In [3]:
from dataclasses import dataclass

@dataclass
class InfrastructureContext:
    cloud: str = "Google Cloud Platform ( GCP )"
    application_platform: str = "kubernetes"
    programming_language: str = "java"

### Setting Up Vector Store & Knowledge Base

In [4]:
def split_with_metadata(docs, metadata):
    splits = text_splitter.split_documents(docs)
    for doc in splits:
        doc.metadata.update(metadata)
    return splits

In [5]:
from langchain_community.document_loaders import TextLoader

loader2 = TextLoader("resources/infra_info.txt")
data2 = loader2.load()

print(data2)

[Document(metadata={'source': 'resources/infra_info.txt'}, page_content='This doc contains infrastructure related information\n\nDeployment-Information\n\nAll Services are deployed in Kubernetes cluster\n\n| Application     | Region   | Deployment          |\n| --------------- | -------- | ------------------- |\n| order-service   | us-east1 | kubernetes-pod-1245 |\n| loyalty-service | us-east2 | kubernetes-pod-3646 |\n| payment-service | us-east1 | kubernetes-pod-3750 |\n')]


In [6]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("resources/incidents.txt")
data = loader.load()

print(data)

[Document(metadata={'source': 'resources/incidents.txt'}, page_content='INCIDENT # 1\n\nTitle: Disk Space Issue in Kubernetes Pods\n\nDescription:\nWe are observing the following warning logs in the beauty-box-service:\n\n2024-03-01 14:22:31 WARNING [Application: beauty-box-service] Disk space low on server prod-db-05. Only 30% remaining. Action required.\n\nImmediate action is required to prevent service degradation and potential outages.\n\nReporter:\nWaqar Mansoor\n\nAssignee:\nSRE-Team - L1\n\nResolution:\nThe issue was caused by excessive usage of space because of the log file. As a temporary mitigation, the affected Kubernetes pods were restarted. A permanent fix will require code-level optimization to control memory usage.\n\nStatus:\nResolved\n\nComments / Timeline:\nSRE-Team - L1: Investigating the issue. Can you confirm the frequency of occurrence?\nWaqar Mansoor: Not exactly sure about the frequency.\nSRE-Team - L1: We will restart the pods and monitor the behavior.\nWaqar M

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)

all_splits = split_with_metadata(data, {
    "doc_type": "incident",
    "source": "prod_incidents"
})

all_splits_2 = split_with_metadata(data2, {
    "doc_type": "infrastructure",
    "source": "k8s_architecture"
})


print("Infra chunks:", len(all_splits))
print("Incident chunks:", len(all_splits_2))

Infra chunks: 8
Incident chunks: 1


In [8]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [9]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [10]:
vector_store.add_documents(all_splits)
vector_store.add_documents(all_splits_2)

['104e9bad-381d-4856-a396-52cb9eeb3e9e']

### Setting Up Tools

In [12]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

db = SQLDatabase.from_uri("sqlite:///resources/Logs.db")

@tool
def query_logs_db(query: str) -> str:

    """Query the database for similar logs structure"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

@tool
def doc_search(doc_type:str, query: str) -> str:
    """Search the data in the docs pass either infrastructure as doc_type for infrastructure related questions or incident as doc_type to search past incidents"""
    results = vector_store.similarity_search(query, filter=lambda doc: doc.metadata.get("doc_type") == doc_type)
    return results[0].page_content

### Setting Up Sub-Agents

In [13]:
from langchain.agents import create_agent

# Log Parser agent
log_parser_agent = create_agent(
    model="gpt-5-nano",
    system_prompt="""
    You are a log parser agent you will be given logs in the raw format and you need to extract these fields from it timestamp, severity level and actual message, you need to format these fields in the proper json format, example: {"timestamp":"2026-02-27T10:00:00Z","application":"payment-service","level":"ERROR","message":message}.
    """
)

# Web Search Agent
web_search_agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    context_schema=InfrastructureContext,
    system_prompt="""
    You are a technical specialist for application log resolution. Search logs and get the top 3 resolutions for the given log.
    You are not allowed to ask any more follow up questions, you must find the best resolutions options based on score, must be greater then 0.96
    You may need to make multiple searches to iteratively find the best options.
    """
)

# DB Search Agent
logs_db_agent = create_agent(
    model="gpt-5-nano",
    tools=[query_logs_db],
    system_prompt="""
    You are a db logs specialist. Query the sql database and find the logs similar to the provided log, you need to find out the frequency of the logs grouped by the app information.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    You may need to make multiple queries to iteratively find the best options.
    Don't return any extra notes in the response just return what you found in the db example "Found the similar error message : <message found in db> in checkout-service at this timestamp frequency was 2
    """
)

# INCIDENT Search Agent
incident_search_agent = create_agent(
    model="gpt-5-nano",
    tools=[doc_search],
    system_prompt="You are a incident search agent, use appropriate tools to search the incidents similar to the provided data logs and return the results"
    )

# Root Cause Analysis Agent
rca_agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search,doc_search],
    context_schema=InfrastructureContext,
    system_prompt="""You are an AI Incident Intelligence Agent responsible for performing structured root cause analysis.
You MUST base your reasoning on:
1. Current parsed log input
2. Statistical logs data
3. Historical incident records
4. Current infrastructure data
5. Context Information

Your task is to:
- Identify the most probable root cause
- Detect recurrence patterns
- Assess severity
- Assign a confidence score (0 to 1)
- Provide actionable remediation steps
- Explicitly reference historical evidence used in reasoning

Example :
Response can follow this template
Incident Title: <Incident Title>
Date & Time Detected: < TimeStamp>
Environment: <Environment>
Affected Host: <Affected Host>
Affected Application:  <affected application>
Severity Level: <Severity Level>
Recommended Action : <Recommended Action>
"""
    )

### Wrapping Sub-Agents under Coordinator Tools

In [14]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command


@tool
async def log_parser(raw_log: str, runtime: ToolRuntime) -> str:
    """Parse logs and converts it into proper formatted json"""
    response = await log_parser_agent.ainvoke({"messages": [HumanMessage(content=f'Parse this log {raw_log}')]})
    return response['messages'][-1].content

@tool
def search_logs(log:str, level:str , runtime: ToolRuntime) -> str:
    """Log search agent chooses the top 3 resolutions for the given log"""
    query = f"Search this log {log} coming as {level}"
    response = web_search_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def search_db(log:str, level:str , runtime: ToolRuntime) -> str:
    """DB search agent use the provided logs and checks the similar logs in the db"""
    query = f"Search this log {log} coming as {level}"
    response = logs_db_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def search_past_incidents(log:str, level:str , runtime: ToolRuntime) -> str:
    """Incident search agent use the provided logs and check the similar incidents happened before"""
    query = f"Search this log {log} coming as {level}"
    response = incident_search_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def root_cause_analysis(formatted_log:str, websearch_results : str, db_results:str, past_incidents: str  , runtime: ToolRuntime) -> str:
    """Analyzes the root cause of the issue, using all the information provided"""
    query = f"Analyze this log : {formatted_log} here are the web search results for this issue : {websearch_results} , here is the statistical log data : {db_results}, here are the similar incidents happened before : {past_incidents} "
    response = rca_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(formatted_log:str, websearch_results: str, db_results:str, past_incidents: str, root_cause_analysis: str, runtime: ToolRuntime) -> str:
    """Update the state when you got all the values"""
    return Command(update={
        "formatted_log": formatted_log,
        "websearch_results" : websearch_results,
        "db_results" : db_results,
        "past_incidents": past_incidents,
        "root_cause_analysis": root_cause_analysis,
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )


### Coordinator Agent Setup

In [17]:
from langchain.agents import create_agent

coordinator = create_agent(
    model="gpt-5-nano",
    tools=[log_parser, search_logs, search_db, search_past_incidents, root_cause_analysis, update_state],
    state_schema=LogState,
    system_prompt="""
    You are a application log coordinator. Delegate tasks to your specialists for raw log parsing and get it in the proper format. Once you get the formatted log take the level and message of it and search the web to get some possible resolutions, check in database also for similar logs, check the similar incidents, once you have all the data perform root cause analysis of the issue, update the state with all data and return the root cause analysis information.
    """
)

### Invoking Coordinator Agent

In [18]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="Here is the log : 2026-03-01 14:22:31 ERROR [Application: OrderService] Disk space critically low on server prod-db-01. Only 2% remaining. Immediate action required.")],
    }
)

In [19]:
print(response["messages"][-1].content)

Root cause analysis for log: "Disk space critically low on server prod-db-01. Only 2% remaining. Immediate action required." (Level: ERROR)

1) Formatted log (from parser)
- timestamp: 2026-03-01T14:22:31Z
- application: OrderService
- level: ERROR
- message: Disk space critically low on server prod-db-01. Only 2% remaining. Immediate action required.

2) Web search resolutions (top recommendations)
- Resolution 1: Free space by cleaning package caches and removing old kernels (Debian/Ubuntu style)
  - Why: Old kernels and caches can consume substantial space; removing unused kernels and cleaning apt caches frees space quickly if current kernel is preserved.
  - Key actions: identify current kernel, list/remove old kernels, autoremove/clean caches, verify with df -h.
- Resolution 2: Vacuum and trim system logs and journal files
  - Why: Journald and rotated logs can dominate disk usage on many servers; trimming while keeping recent data reduces space without turning off logging.
  - Ke

In [20]:
from pprint import pprint

pprint(response)

{'db_results': '',
 'formatted_log': '',
 'messages': [HumanMessage(content='Here is the log : 2026-03-01 14:22:31 ERROR [Application: OrderService] Disk space critically low on server prod-db-01. Only 2% remaining. Immediate action required.', additional_kwargs={}, response_metadata={}, id='f18b121a-8ba9-4f63-abdb-276dcf652c35'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 575, 'prompt_tokens': 469, 'total_tokens': 1044, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DFRQvmq1uZpKRcXMV3hAX93NkoRT2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cb591-9732-7482-b469-96959ee68a0f-0', tool_cal